In [2]:
import os
import polars as pl
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import train_test_split

from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, mean_absolute_percentage_error

In [3]:
WORKING_DIR = '/group/pmc021/amunif/epi-thesis/workflow/08_HepG2/'

In [4]:
def load_data(path):
    gene_pl = pd.read_parquet(path)
    return gene_pl

In [5]:
def reformat_data(df, columns):
    processed_arrays = []

    for col in columns:
        stacked = np.vstack(df[col].values)
        processed_arrays.append(stacked)

    X = np.hstack(processed_arrays)
    return X

In [6]:
# Load dataset
gene_pl = load_data(os.path.join(WORKING_DIR, 'dataset', 'gene_w_label_value_1.parquet'))
gene_pl.head(5)

,gene_id,H3K4me3,H3K4me3_count,H3K9ac,H3K9ac_count,H3K9me3,H3K9me3_count,H3K27ac,H3K27ac_count,H3K27me3,H3K27me3_count,value_1,label
0,XLOC_000001,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,0.000000,0
1,XLOC_000003,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,0.000000,0
2,XLOC_000006,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,0.088845,0
3,XLOC_000007,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,4.047430,1
4,XLOC_000008,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",6,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",3,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",2,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0,26.793400,1


In [7]:
gene_pl["value_1"].describe()

count    22154.000000
mean        21.856847
std        167.605195
min          0.000000
25%          0.015560
50%          1.142095
75%         12.266025
max      12870.900000
Name: value_1, dtype: float64

In [8]:
markers = ['H3K4me3', 'H3K9ac', 'H3K9me3', 'H3K27ac', 'H3K27me3']
X = reformat_data(gene_pl, markers)

In [9]:
print(X)
print(X.shape)

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
(22154, 20000)


In [11]:
y = gene_pl['value_1'].values
print(y)
print(y.shape)

[0.        0.        0.0888452 ... 0.        0.        0.       ]
(22154,)


In [12]:
random_state = 42

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)

In [13]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

(17723, 20000)
(4431, 20000)
(17723,)
(4431,)


In [20]:
# Create MLPRegressor model
mlp = MLPRegressor(hidden_layer_sizes=(256, 128, 64), 
                   activation='relu', 
                   solver='adam', 
                   alpha=0.0001, 
                   max_iter=1000, 
                   random_state=42,
                   verbose=True)

In [21]:
# Train the model
mlp.fit(X_train, y_train)

Iteration 1, loss = 16068.59931845
Iteration 2, loss = 15973.70150621
Iteration 3, loss = 15854.60549334
Iteration 4, loss = 15606.30644019
Iteration 5, loss = 15174.10068908
Iteration 6, loss = 14844.65275261
Iteration 7, loss = 14358.40516714
Iteration 8, loss = 13562.70784446
Iteration 9, loss = 11993.44659978
Iteration 10, loss = 12476.55694562
Iteration 11, loss = 11792.23827528
Iteration 12, loss = 11086.97876858
Iteration 13, loss = 10126.96603587
Iteration 14, loss = 9319.91116460
Iteration 15, loss = 9886.07495867
Iteration 16, loss = 9422.73361217
Iteration 17, loss = 8796.54650977
Iteration 18, loss = 8811.76406431
Iteration 19, loss = 8407.51681127
Iteration 20, loss = 8270.03282545
Iteration 21, loss = 8484.20465982
Iteration 22, loss = 8306.28518387
Iteration 23, loss = 7548.03888807
Iteration 24, loss = 7454.11909613
Iteration 25, loss = 7607.65298550
Iteration 26, loss = 7368.38104880
Iteration 27, loss = 7556.00389855
Iteration 28, loss = 7517.12712198
Iteration 29, lo

MLPRegressor(hidden_layer_sizes=(256, 128, 64), max_iter=1000, random_state=42,
             verbose=True)

In [22]:
# Make predictions on the test set
y_pred = mlp.predict(X_test)

# Calculate Mean Squared Error and R-squared score
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse:.4f}")
print(f"R-squared Score: {r2:.4f}")

Mean Squared Error: 63550.9516
R-squared Score: -4.5944
